# اليوم الثالث — مختبر 5: البحث الدلالي | Day 3 — Lab 5: Semantic Search

**المدربة / Instructor:** ميعاد المري — Meaad Al-Marri  
**المسار:** 🟢 Core → 🔵 Explore → 🟣 Distinction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/almiyead-rgb/bayan-applied-nlp-course/blob/main/notebooks/06_semantic_search.ipynb)

**الهدف:** بناء بحث عربي/إنجليزي فعلي: نص → sentence embeddings → L2 normalisation → `FAISS IndexFlatIP` → ترتيب → Recall@3 وMRR@3.

**Goal:** build and evaluate an actual bilingual semantic index. No planted vectors and no API key are used.


## عقد المختبر | Lab contract

- النموذج: `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`؛ راجع model card والترخيص قبل استخدام آخر.
- Core يعمل على CPU؛ GPU اختياري وغير مضمون في Colab Free.
- لا توجد vector database مدفوعة. الفهرس محلي في runtime.
- corpus وquery يجب أن يتبعا **نفس** preprocessing وL2 contract.
- نضبط no-answer threshold على `validation` فقط، ثم نجمّده قبل `test`.
- القياسات على عينة الدورة الصغيرة تسمى `MEASURED_SMOKE` وليست نتيجة إنتاجية.


In [1]:
import importlib.metadata
import subprocess
import sys

REQUIRED = {
    "camel-tools": "1.6.0",
    "sentence-transformers": "6.0.0",
    "faiss-cpu": "1.15.0",
    "transformers": "5.15.1",
    "tokenizers": "0.22.2",
}
to_install = []
for distribution, expected in REQUIRED.items():
    try:
        current = importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        current = None
    if current != expected:
        to_install.append(f"{distribution}=={expected}")
if to_install:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", *to_install]
    )
for distribution, expected in REQUIRED.items():
    assert importlib.metadata.version(distribution) == expected
print("SETUP=PASS", {name: importlib.metadata.version(name) for name in REQUIRED})


SETUP=PASS {'camel-tools': '1.6.0', 'sentence-transformers': '6.0.0', 'faiss-cpu': '1.15.0', 'transformers': '5.15.1', 'tokenizers': '0.22.2'}


In [2]:
import csv
import hashlib
import io
import json
import urllib.request
import statistics
import time
from collections import defaultdict
from pathlib import Path

import faiss
import numpy as np
import torch
from camel_tools.utils.dediac import dediac_ar
from camel_tools.utils.normalize import (
    normalize_alef_ar,
    normalize_alef_maksura_ar,
    normalize_unicode,
)
from sentence_transformers import SentenceTransformer

DATA_KIND = "MEASURED_SMOKE"
MODEL_ID = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
K = 3
print("DEVICE", "cuda" if torch.cuda.is_available() else "cpu")


DEVICE cpu


## 1) corpus وqueries موثقة | Documented corpus and queries

الحالات والاستعلامات اصطناعية. يوجد `relevant_case_ids` معلوم لكل query قابلة للإجابة، وarray فارغة لحالات no-answer. النسخة المدمجة مطابقة للملفين العامين وتستخدم فقط عند تعذر الرابط.


In [3]:
CASES_URL = "https://raw.githubusercontent.com/almiyead-rgb/bayan-applied-nlp-course/main/data/sample/bayan_day3_cases.csv"
QUERIES_URL = "https://raw.githubusercontent.com/almiyead-rgb/bayan-applied-nlp-course/main/data/sample/bayan_day3_queries.jsonl"
CASES_FALLBACK = [{'case_id': 'AR-001', 'language': 'ar', 'variant': 'MSA', 'topic': 'digital_service', 'summary': 'تعذر تسجيل الدخول إلى البوابة بعد تحديث كلمة المرور', 'resolution': 'تمت إعادة مزامنة الحساب وإرسال رابط دخول جديد'}, {'case_id': 'AR-002', 'language': 'ar', 'variant': 'Gulf', 'topic': 'digital_service', 'summary': 'ما وصل رمز التحقق للجوال عند محاولة الدخول', 'resolution': 'تم تحديث رقم التواصل وإعادة إرسال الرمز'}, {'case_id': 'AR-003', 'language': 'ar', 'variant': 'MSA', 'topic': 'digital_service', 'summary': 'يفشل رفع ملف PDF في صفحة الطلب', 'resolution': 'تم ضغط الملف وتغيير اسمه ثم اكتمل الرفع'}, {'case_id': 'AR-004', 'language': 'ar', 'variant': 'Gulf', 'topic': 'transport', 'summary': 'الباص تأخر عن المحطة أكثر من نصف ساعة', 'resolution': 'تمت إضافة رحلة بديلة وإشعار المستفيدين'}, {'case_id': 'AR-005', 'language': 'ar', 'variant': 'MSA', 'topic': 'transport', 'summary': 'لا يظهر مسار الحافلة الجديد في التطبيق', 'resolution': 'تم تحديث بيانات المسار وإعادة تحميل الخريطة'}, {'case_id': 'AR-006', 'language': 'ar', 'variant': 'Gulf', 'topic': 'transport', 'summary': 'موقع موقف الحافلة غير واضح في الحي', 'resolution': 'تم إرسال رابط الموقع وإضافة لوحة إرشادية'}, {'case_id': 'AR-007', 'language': 'ar', 'variant': 'MSA', 'topic': 'health', 'summary': 'لا توجد مواعيد متاحة في العيادة المطلوبة', 'resolution': 'تم فتح قائمة انتظار واقتراح عيادة قريبة'}, {'case_id': 'AR-008', 'language': 'ar', 'variant': 'Gulf', 'topic': 'health', 'summary': 'نتيجة التحليل ما ظهرت في التطبيق الصحي', 'resolution': 'تمت مزامنة النتيجة مع الملف الصحي'}, {'case_id': 'AR-009', 'language': 'ar', 'variant': 'MSA', 'topic': 'health', 'summary': 'تعذر تجديد الوصفة من خلال التطبيق', 'resolution': 'تم التحقق من الأهلية وإعادة تفعيل طلب التجديد'}, {'case_id': 'AR-010', 'language': 'ar', 'variant': 'MSA', 'topic': 'permit', 'summary': 'رُفض المستند المرفق بطلب التصريح', 'resolution': 'تم توضيح صيغة المستند وإعادة فتح الطلب للرفع'}, {'case_id': 'AR-011', 'language': 'ar', 'variant': 'Gulf', 'topic': 'permit', 'summary': 'طلب التصريح واقف عند المراجعة من أسبوع', 'resolution': 'تم تصعيد الطلب وتحديث الحالة في اليوم التالي'}, {'case_id': 'AR-012', 'language': 'ar', 'variant': 'MSA', 'topic': 'permit', 'summary': 'تم احتساب رسوم التصريح مرتين', 'resolution': 'أعيد المبلغ المكرر وثُبتت عملية دفع واحدة'}, {'case_id': 'EN-001', 'language': 'en', 'variant': 'English', 'topic': 'digital_service', 'summary': 'Cannot sign in to the portal after changing the password', 'resolution': 'The account was resynchronised and a new sign-in link was sent'}, {'case_id': 'EN-002', 'language': 'en', 'variant': 'English', 'topic': 'digital_service', 'summary': 'The verification code never arrived on the registered phone', 'resolution': 'The contact number was verified and the code was resent'}, {'case_id': 'EN-003', 'language': 'en', 'variant': 'English', 'topic': 'digital_service', 'summary': 'The portal rejects a PDF attachment during upload', 'resolution': 'The file was compressed and renamed before a successful upload'}, {'case_id': 'EN-004', 'language': 'en', 'variant': 'English', 'topic': 'transport', 'summary': 'The bus arrived more than thirty minutes late', 'resolution': 'An additional trip was assigned and passengers were notified'}, {'case_id': 'EN-005', 'language': 'en', 'variant': 'English', 'topic': 'transport', 'summary': 'The new bus route is missing from the mobile map', 'resolution': 'The route dataset was refreshed and the map was reloaded'}, {'case_id': 'EN-006', 'language': 'en', 'variant': 'English', 'topic': 'transport', 'summary': 'The location of the neighbourhood bus stop is unclear', 'resolution': 'A location link was sent and a sign was added'}, {'case_id': 'EN-007', 'language': 'en', 'variant': 'English', 'topic': 'health', 'summary': 'No appointments are available at the requested clinic', 'resolution': 'A waiting list was opened and a nearby clinic was suggested'}, {'case_id': 'EN-008', 'language': 'en', 'variant': 'English', 'topic': 'health', 'summary': 'The laboratory result is missing from the health application', 'resolution': 'The result was synchronised with the health record'}, {'case_id': 'EN-009', 'language': 'en', 'variant': 'English', 'topic': 'health', 'summary': 'The prescription renewal action fails in the application', 'resolution': 'Eligibility was checked and the renewal request was reactivated'}, {'case_id': 'EN-010', 'language': 'en', 'variant': 'English', 'topic': 'permit', 'summary': 'The permit request rejected the uploaded document', 'resolution': 'The required format was explained and upload was reopened'}, {'case_id': 'EN-011', 'language': 'en', 'variant': 'English', 'topic': 'permit', 'summary': 'The permit status has remained under review for a week', 'resolution': 'The request was escalated and its status was updated the next day'}, {'case_id': 'EN-012', 'language': 'en', 'variant': 'English', 'topic': 'permit', 'summary': 'The permit fee was charged twice', 'resolution': 'The duplicate amount was refunded and one payment was retained'}]
QUERIES_FALLBACK = [{'query_id': 'QV-001', 'split': 'validation', 'query': 'ما وصلني كود الدخول على الجوال', 'language': 'ar', 'retrieval_mode': 'monolingual', 'relevant_case_ids': ['AR-002']}, {'query_id': 'QV-002', 'split': 'validation', 'query': 'The portal will not accept my PDF file', 'language': 'en', 'retrieval_mode': 'monolingual', 'relevant_case_ids': ['EN-003']}, {'query_id': 'QV-003', 'split': 'validation', 'query': 'The bus was over thirty minutes late', 'language': 'en', 'retrieval_mode': 'cross_lingual', 'relevant_case_ids': ['AR-004']}, {'query_id': 'QV-004', 'split': 'validation', 'query': 'انخصمت رسوم التصريح مرتين', 'language': 'ar', 'retrieval_mode': 'cross_lingual', 'relevant_case_ids': ['EN-012']}, {'query_id': 'QV-005', 'split': 'validation', 'query': 'أحتاج موعد عيادة ولا يوجد وقت متاح', 'language': 'ar', 'retrieval_mode': 'monolingual', 'relevant_case_ids': ['AR-007']}, {'query_id': 'QV-006', 'split': 'validation', 'query': 'My permit has been under review for a week', 'language': 'en', 'retrieval_mode': 'monolingual', 'relevant_case_ids': ['EN-011']}, {'query_id': 'QV-007', 'split': 'validation', 'query': 'ما ساعات عمل المكتبة؟', 'language': 'ar', 'retrieval_mode': 'no_answer', 'relevant_case_ids': []}, {'query_id': 'QV-008', 'split': 'validation', 'query': "What is tomorrow's weather?", 'language': 'en', 'retrieval_mode': 'no_answer', 'relevant_case_ids': []}, {'query_id': 'QV-009', 'split': 'validation', 'query': 'أرغب في التقديم على وظيفة', 'language': 'ar', 'retrieval_mode': 'no_answer', 'relevant_case_ids': []}, {'query_id': 'QV-010', 'split': 'validation', 'query': 'Where can I renew my passport?', 'language': 'en', 'retrieval_mode': 'no_answer', 'relevant_case_ids': []}, {'query_id': 'QT-001', 'split': 'test', 'query': 'رمز التحقق لا يصل لهاتفي', 'language': 'ar', 'retrieval_mode': 'cross_lingual', 'relevant_case_ids': ['EN-002']}, {'query_id': 'QT-002', 'split': 'test', 'query': 'The new route is absent from the map', 'language': 'en', 'retrieval_mode': 'monolingual', 'relevant_case_ids': ['EN-005']}, {'query_id': 'QT-003', 'split': 'test', 'query': 'نتيجة المختبر غير موجودة في الملف الصحي', 'language': 'ar', 'retrieval_mode': 'monolingual', 'relevant_case_ids': ['AR-008']}, {'query_id': 'QT-004', 'split': 'test', 'query': 'The uploaded permit document was refused', 'language': 'en', 'retrieval_mode': 'cross_lingual', 'relevant_case_ids': ['AR-010']}, {'query_id': 'QT-005', 'split': 'test', 'query': 'وين موقع موقف الباص في الحي؟', 'language': 'ar', 'retrieval_mode': 'monolingual', 'relevant_case_ids': ['AR-006']}, {'query_id': 'QT-006', 'split': 'test', 'query': 'I cannot renew my prescription in the app', 'language': 'en', 'retrieval_mode': 'monolingual', 'relevant_case_ids': ['EN-009']}, {'query_id': 'QT-007', 'split': 'test', 'query': 'أحتاج وصفة طبخ سريعة', 'language': 'ar', 'retrieval_mode': 'no_answer', 'relevant_case_ids': []}, {'query_id': 'QT-008', 'split': 'test', 'query': 'How do I reserve a football field?', 'language': 'en', 'retrieval_mode': 'no_answer', 'relevant_case_ids': []}]

def load_course_data():
    try:
        with urllib.request.urlopen(CASES_URL, timeout=15) as response:
            cases = list(csv.DictReader(io.StringIO(response.read().decode("utf-8"))))
        with urllib.request.urlopen(QUERIES_URL, timeout=15) as response:
            queries = [json.loads(line) for line in response.read().decode("utf-8").splitlines() if line.strip()]
        return cases, queries, "github"
    except Exception as exc:
        return CASES_FALLBACK, QUERIES_FALLBACK, f"embedded_fallback:{type(exc).__name__}"

cases, queries, data_source = load_course_data()
assert len(cases) == 24 and len(queries) == 18
assert len({row["case_id"] for row in cases}) == len(cases)
assert {row["split"] for row in queries} == {"validation", "test"}
assert all(isinstance(row["relevant_case_ids"], list) for row in queries)
print({"source": data_source, "cases": len(cases), "queries": len(queries)})


{'source': 'github', 'cases': 24, 'queries': 18}


## 2) preprocessing متطابق | Symmetric preprocessing

العربية تتبع `search profile 1.0.0` من Notebook 05. الإنجليزية تستخدم Unicode NFC والمسافات فقط. نطبق الدالة نفسها على corpus وquery؛ وإلا يصبح القياس بين فضاءين مختلفين.


In [4]:
def normalise_for_search(text, language):
    text = normalize_unicode(text, compatibility=False)
    text = " ".join(text.replace("ـ", "").split())
    if language == "ar":
        text = dediac_ar(text)
        text = normalize_alef_ar(text)
        text = normalize_alef_maksura_ar(text)
    return text

corpus_ids = [row["case_id"] for row in cases]
corpus_texts = [
    normalise_for_search(f'{row["summary"]} [SEP] {row["resolution"]}', row["language"])
    for row in cases
]
assert len(corpus_ids) == len(corpus_texts) == 24
print(corpus_ids[0], corpus_texts[0])


AR-001 تعذر تسجيل الدخول الي البوابة بعد تحديث كلمة المرور [SEP] تمت اعادة مزامنة الحساب وارسال رابط دخول جديد


## 3) Sentence embedding حقيقي | Actual model embeddings

الـbi-encoder يرمّز الوثائق مسبقًا ثم يرمّز كل query مرة. هذا يختلف عن token embeddings وعن logits المصنف. أول تشغيل ينزل weights من Hugging Face؛ إذا فشل التنزيل افحص الاتصال مرة واحدة، ثم انتقل إلى Notebook 07 ولا تنشئ vectors عشوائية.


In [5]:
try:
    model = SentenceTransformer(MODEL_ID)
except Exception as exc:
    raise RuntimeError(
        "تعذر تنزيل نموذج البحث. افحص اتصال Hugging Face ثم أعد هذه الخلية مرة واحدة؛ "
        "لا تستخدم vectors عشوائية كبديل لـCore."
    ) from exc

corpus_vectors = model.encode(
    corpus_texts,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
).astype("float32")
vector_norms = np.linalg.norm(corpus_vectors, axis=1)
assert corpus_vectors.shape[0] == len(cases)
assert corpus_vectors.shape[1] > 0
assert np.allclose(vector_norms, 1.0, atol=1e-5)
print({"shape": corpus_vectors.shape, "norm_min": float(vector_norms.min()), "norm_max": float(vector_norms.max())})


{'shape': (24, 384), 'norm_min': 0.9999999403953552, 'norm_max': 1.0000001192092896}


## 4) FAISS IndexFlatIP

بعد L2 normalisation، inner product يرتب بالاتجاه نفسه الذي يرتب به cosine similarity. `IndexFlatIP` exact baseline: بسيط وقابل للتحقق ومناسب لـ24 حالة. ANN يصبح قرارًا له معنى عند حجم أكبر وبعد مقارنة الجودة والسرعة.


In [6]:
dimension = corpus_vectors.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(corpus_vectors)
assert index.ntotal == len(cases)
print({"index_type": type(index).__name__, "vectors": index.ntotal, "dimension": dimension})


{'index_type': 'IndexFlatIP', 'vectors': 24, 'dimension': 384}


In [7]:
def search(query, language, k=K):
    clean_query = normalise_for_search(query, language)
    query_vector = model.encode(
        [clean_query], convert_to_numpy=True, normalize_embeddings=True
    ).astype("float32")
    assert np.allclose(np.linalg.norm(query_vector, axis=1), 1.0, atol=1e-5)
    scores, positions = index.search(query_vector, min(k, len(cases)))
    return [
        {
            "rank": rank,
            "case_id": corpus_ids[position],
            "score": float(score),
            "summary": cases[position]["summary"],
            "language": cases[position]["language"],
        }
        for rank, (position, score) in enumerate(zip(positions[0], scores[0]), start=1)
    ]

for demo_query, language in [
    ("رمز الدخول لم يصل إلى جوالي", "ar"),
    ("The bus route is absent from the map", "en"),
    ("تم خصم رسوم التصريح مرتين", "ar"),
]:
    print("\nQUERY:", demo_query)
    for result in search(demo_query, language):
        print(result)



QUERY: رمز الدخول لم يصل إلى جوالي
{'rank': 1, 'case_id': 'AR-001', 'score': 0.5912885069847107, 'summary': 'تعذر تسجيل الدخول إلى البوابة بعد تحديث كلمة المرور', 'language': 'ar'}
{'rank': 2, 'case_id': 'EN-001', 'score': 0.5889512300491333, 'summary': 'Cannot sign in to the portal after changing the password', 'language': 'en'}
{'rank': 3, 'case_id': 'AR-002', 'score': 0.5306751132011414, 'summary': 'ما وصل رمز التحقق للجوال عند محاولة الدخول', 'language': 'ar'}

QUERY: The bus route is absent from the map
{'rank': 1, 'case_id': 'EN-005', 'score': 0.7517009973526001, 'summary': 'The new bus route is missing from the mobile map', 'language': 'en'}
{'rank': 2, 'case_id': 'AR-005', 'score': 0.6912578344345093, 'summary': 'لا يظهر مسار الحافلة الجديد في التطبيق', 'language': 'ar'}
{'rank': 3, 'case_id': 'AR-006', 'score': 0.6660596132278442, 'summary': 'موقع موقف الحافلة غير واضح في الحي', 'language': 'ar'}

QUERY: تم خصم رسوم التصريح مرتين
{'rank': 1, 'case_id': 'AR-012', 'score': 0.59

### تحقق يدوي | Manual check

اقرأ النصوص لا الأرقام فقط:

- هل أول نتيجة تجيب المعنى؟
- هل ظهر النظير باللغة الأخرى ضمن أول 3 عند الاستعلام cross-lingual؟
- هل درجتان متقاربتان تعنيان بالضرورة أن النتيجتين صحيحتان؟ لا.


## 5) قياس الاسترجاع | Retrieval evaluation

- `Recall@3`: نسبة queries القابلة للإجابة التي ظهر صحيح لها في أول 3.
- `MRR@3`: يعطي وزنًا أكبر عندما يظهر أول صحيح في رتبة مبكرة.
- no-answer يقاس منفصلًا؛ لا ندخل query بلا relevant case في Recall/MRR.


In [8]:
def retrieval_metrics(ranked_ids, relevant_ids, k=3):
    hits, reciprocal_ranks = [], []
    for ranking, relevant in zip(ranked_ids, relevant_ids):
        relevant = set(relevant)
        if not relevant:
            continue
        first = next((rank for rank, item in enumerate(ranking[:k], 1) if item in relevant), None)
        hits.append(float(first is not None))
        reciprocal_ranks.append(1.0 / first if first else 0.0)
    if not hits:
        raise ValueError("at least one answerable query is required")
    return {
        f"recall@{k}": float(np.mean(hits)),
        f"mrr@{k}": float(np.mean(reciprocal_ranks)),
        "answerable_queries": len(hits),
    }

def rank_queries(query_subset):
    rows = []
    for query in query_subset:
        ranking = search(query["query"], query["language"], k=K)
        rows.append({
            **query,
            "ranked_case_ids": [item["case_id"] for item in ranking],
            "best_score": ranking[0]["score"],
        })
    return rows

validation_rankings = rank_queries([row for row in queries if row["split"] == "validation"])
test_rankings = rank_queries([row for row in queries if row["split"] == "test"])
print("ranked", len(validation_rankings), "validation and", len(test_rankings), "test queries")


ranked 10 validation and 8 test queries


## 6) no-answer threshold: Validation ثم Test

نعطي نتيجة إذا كان `best_score >= threshold`. نختار threshold الذي يزيد دقة answer/no-answer على validation فقط. ثم لا نغيّره بعد رؤية test.


In [9]:
def tune_no_answer_threshold(best_scores, has_relevant):
    scores = np.asarray(best_scores, dtype=float)
    labels = np.asarray(has_relevant, dtype=bool)
    unique = sorted(set(float(score) for score in scores))
    candidates = [unique[0] - 1e-6]
    candidates += [(left + right) / 2 for left, right in zip(unique, unique[1:])]
    candidates += [unique[-1] + 1e-6]
    scored = []
    for threshold in candidates:
        accuracy = float(np.mean((scores >= threshold) == labels))
        scored.append((accuracy, threshold))
    accuracy, threshold = max(scored, key=lambda item: (item[0], item[1]))
    return {"threshold": float(threshold), "validation_accuracy": accuracy}

threshold_result = tune_no_answer_threshold(
    [row["best_score"] for row in validation_rankings],
    [bool(row["relevant_case_ids"]) for row in validation_rankings],
)
FROZEN_THRESHOLD = threshold_result["threshold"]
print("VALIDATION_ONLY", threshold_result)

test_no_answer_accuracy = float(np.mean([
    (row["best_score"] >= FROZEN_THRESHOLD) == bool(row["relevant_case_ids"])
    for row in test_rankings
]))
print("TEST_WITH_FROZEN_THRESHOLD", {"no_answer_accuracy": test_no_answer_accuracy})


VALIDATION_ONLY {'threshold': 0.4592096209526062, 'validation_accuracy': 1.0}
TEST_WITH_FROZEN_THRESHOLD {'no_answer_accuracy': 1.0}


In [10]:
answerable_test = [row for row in test_rankings if row["relevant_case_ids"]]
overall_metrics = retrieval_metrics(
    [row["ranked_case_ids"] for row in answerable_test],
    [row["relevant_case_ids"] for row in answerable_test],
    k=K,
)

slice_metrics = []
for key in ["language", "retrieval_mode"]:
    for value in sorted({row[key] for row in answerable_test}):
        group = [row for row in answerable_test if row[key] == value]
        metric = retrieval_metrics(
            [row["ranked_case_ids"] for row in group],
            [row["relevant_case_ids"] for row in group],
            k=K,
        )
        slice_metrics.append({"slice": f"{key}={value}", "n": len(group), "flag": "SMALL_SLICE" if len(group) < 10 else "", **metric})

print("MEASURED_SMOKE overall", overall_metrics)
for row in slice_metrics:
    print("MEASURED_SMOKE", row)


MEASURED_SMOKE overall {'recall@3': 1.0, 'mrr@3': 0.6666666666666666, 'answerable_queries': 6}
MEASURED_SMOKE {'slice': 'language=ar', 'n': 3, 'flag': 'SMALL_SLICE', 'recall@3': 1.0, 'mrr@3': 0.5, 'answerable_queries': 3}
MEASURED_SMOKE {'slice': 'language=en', 'n': 3, 'flag': 'SMALL_SLICE', 'recall@3': 1.0, 'mrr@3': 0.8333333333333334, 'answerable_queries': 3}
MEASURED_SMOKE {'slice': 'retrieval_mode=cross_lingual', 'n': 2, 'flag': 'SMALL_SLICE', 'recall@3': 1.0, 'mrr@3': 0.5, 'answerable_queries': 2}
MEASURED_SMOKE {'slice': 'retrieval_mode=monolingual', 'n': 4, 'flag': 'SMALL_SLICE', 'recall@3': 1.0, 'mrr@3': 0.75, 'answerable_queries': 4}


## 7) manifest: ما الذي بنى الفهرس؟ | Index provenance

حفظ `.index` وحده لا يكفي. الـmanifest يربط vectors بالنموذج وpreprocessing والبيانات والبعد. أي تغيير في هذه الحقول يستلزم إعادة البناء.


In [11]:
dataset_bytes = json.dumps(cases, ensure_ascii=False, sort_keys=True).encode("utf-8")
manifest = {
    "manifest_version": "1.0.0",
    "model_id": MODEL_ID,
    "embedding_dimension": int(dimension),
    "normalization": "l2",
    "preprocessing_profile": "arabic-search/1.0.0 + english-nfc-whitespace/1.0.0",
    "dataset_id": "bayan_day3_cases.csv",
    "dataset_sha256": hashlib.sha256(dataset_bytes).hexdigest(),
    "vector_count": int(index.ntotal),
    "index_type": "IndexFlatIP",
    "data_kind": DATA_KIND,
    "libraries": {name: importlib.metadata.version(name) for name in REQUIRED},
}
retrieval_report = {
    "data_kind": DATA_KIND,
    "k": K,
    "threshold_tuned_on": "validation",
    "frozen_no_answer_threshold": FROZEN_THRESHOLD,
    "validation_no_answer_accuracy": threshold_result["validation_accuracy"],
    "test_no_answer_accuracy": test_no_answer_accuracy,
    "test_retrieval": overall_metrics,
    "test_slices": slice_metrics,
}

reports_dir = Path("reports")
reports_dir.mkdir(exist_ok=True)
(reports_dir / "search_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
(reports_dir / "retrieval_metrics.json").write_text(json.dumps(retrieval_report, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(manifest, ensure_ascii=False, indent=2))


{
  "manifest_version": "1.0.0",
  "model_id": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
  "embedding_dimension": 384,
  "normalization": "l2",
  "preprocessing_profile": "arabic-search/1.0.0 + english-nfc-whitespace/1.0.0",
  "dataset_id": "bayan_day3_cases.csv",
  "dataset_sha256": "7708cbe884a3c268d24ed2cb87ad2f0a8b64b2e6fa6b37a32393b6ae3bd50e5b",
  "vector_count": 24,
  "index_type": "IndexFlatIP",
  "data_kind": "MEASURED_SMOKE",
  "libraries": {
    "camel-tools": "1.6.0",
    "sentence-transformers": "6.0.0",
    "faiss-cpu": "1.15.0",
    "transformers": "5.15.1",
    "tokenizers": "0.22.2"
  }
}


## 8) Core: retrieve ثم re-rank

الـCrossEncoder يقرأ `(query, candidate)` معًا؛ استخدمه على top-k صغير فقط. تنفذ الخلية التالية بعد الاسترجاع الأولي. لا تفترض أنه أفضل: قس Recall/MRR والlatency قبل القرار.


In [12]:
from sentence_transformers import CrossEncoder
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()
RERANKER_ID = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
RERANK_CANDIDATES = 6
reranker = CrossEncoder(RERANKER_ID)

def rerank_one(query_row, *, measure=True):
    candidates = search(query_row["query"], query_row["language"], k=RERANK_CANDIDATES)
    case_lookup = {row["case_id"]: row for row in cases}
    pairs = [
        (query_row["query"], f'{case_lookup[item["case_id"]]["summary"]} {case_lookup[item["case_id"]]["resolution"]}')
        for item in candidates
    ]
    started = time.perf_counter()
    scores = reranker.predict(pairs, show_progress_bar=False)
    elapsed_ms = (time.perf_counter() - started) * 1000.0
    reranked = sorted(
        zip([item["case_id"] for item in candidates], scores),
        key=lambda item: float(item[1]), reverse=True,
    )
    return {
        **query_row,
        "before_ids": [item["case_id"] for item in candidates],
        "after_ids": [case_id for case_id, _ in reranked],
        "rerank_ms": elapsed_ms if measure else None,
    }

# Warm-up is excluded from latency, then the frozen answerable test slice is measured.
_ = rerank_one(next(row for row in validation_rankings if row["relevant_case_ids"]), measure=False)
reranked_test = [rerank_one(row) for row in answerable_test]
before_rerank = retrieval_metrics(
    [row["before_ids"] for row in reranked_test],
    [row["relevant_case_ids"] for row in reranked_test], k=K,
)
after_rerank = retrieval_metrics(
    [row["after_ids"] for row in reranked_test],
    [row["relevant_case_ids"] for row in reranked_test], k=K,
)
rerank_latencies = [row["rerank_ms"] for row in reranked_test]
rerank_report = {
    "result_type": DATA_KIND,
    "reranker_id": RERANKER_ID,
    "candidate_count": RERANK_CANDIDATES,
    "answerable_test_queries": len(reranked_test),
    "mrr_at_3_before": before_rerank["mrr@3"],
    "mrr_at_3_after": after_rerank["mrr@3"],
    "mrr_at_3_delta": after_rerank["mrr@3"] - before_rerank["mrr@3"],
    "median_rerank_ms": float(statistics.median(rerank_latencies)),
    "p95_rerank_ms": float(np.percentile(rerank_latencies, 95)),
    "warmup_excluded": True,
    "decision": "ADOPT_FOR_EXPERIMENT" if after_rerank["mrr@3"] > before_rerank["mrr@3"] else "REJECT_NO_MEASURED_LIFT",
    "limitations": ["six answerable test queries", "CPU timing depends on runtime", "MEASURED_SMOKE only"],
}
retrieval_report["reranking"] = rerank_report
(reports_dir / "retrieval_metrics.json").write_text(
    json.dumps(retrieval_report, ensure_ascii=False, indent=2), encoding="utf-8"
)
rerank_display = {
    key: rerank_report[key]
    for key in [
        "result_type", "reranker_id", "candidate_count",
        "answerable_test_queries", "mrr_at_3_before",
        "mrr_at_3_after", "mrr_at_3_delta",
        "warmup_excluded", "decision",
    ]
}
rerank_display["latency_note"] = "measured on CPU; runtime-dependent"
print(json.dumps(rerank_display, ensure_ascii=False, indent=2))
print("RERANKING_TRADEOFF=MEASURED_SMOKE")

{
  "result_type": "MEASURED_SMOKE",
  "reranker_id": "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",
  "candidate_count": 6,
  "answerable_test_queries": 6,
  "mrr_at_3_before": 0.6666666666666666,
  "mrr_at_3_after": 0.7222222222222222,
  "mrr_at_3_delta": 0.05555555555555558,
  "warmup_excluded": true,
  "decision": "ADOPT_FOR_EXPERIMENT",
  "latency_note": "measured on CPU; runtime-dependent"
}
RERANKING_TRADEOFF=MEASURED_SMOKE


## بوابة Core | Core gate

العلامة التالية تثبت أن embeddings جاءت من النموذج الفعلي، وأن الجهتين مطبعتان، وأن index/manifest/metrics متسقة. لا تعدّل نصها يدويًا.


In [13]:
core_checks = {
    "actual_sentence_model": getattr(model, "encode", None) is not None,
    "reranking_measured": len(reranked_test) > 0 and rerank_report["warmup_excluded"],
    "all_corpus_vectors_created": corpus_vectors.shape[0] == len(cases),
    "l2_normalised": bool(np.allclose(np.linalg.norm(corpus_vectors, axis=1), 1.0, atol=1e-5)),
    "faiss_count_matches": int(index.ntotal) == len(cases),
    "manifest_matches": manifest["vector_count"] == int(index.ntotal) and manifest["embedding_dimension"] == dimension,
    "validation_only_threshold": retrieval_report["threshold_tuned_on"] == "validation",
    "metrics_in_range": all(0.0 <= overall_metrics[key] <= 1.0 for key in ["recall@3", "mrr@3"]),
    "reports_written": all((reports_dir / name).exists() for name in ["search_manifest.json", "retrieval_metrics.json"]),
}
assert all(core_checks.values()), core_checks
print(core_checks)
print("DAY3_NOTEBOOK6_CORE=PASS")


{'actual_sentence_model': True, 'reranking_measured': True, 'all_corpus_vectors_created': True, 'l2_normalised': True, 'faiss_count_matches': True, 'manifest_matches': True, 'validation_only_threshold': True, 'metrics_in_range': True, 'reports_written': True}
DAY3_NOTEBOOK6_CORE=PASS


## بعد المختبر | After the lab

1. احفظ `reports/search_manifest.json` و`reports/retrieval_metrics.json`.
2. لا ترفع model cache أو weights أو `.faiss` كبيرًا للمستودع.
3. وثق النموذج والترخيص وk والthreshold في `DECISIONS.md`.
4. استخدم commit: `feat: build bilingual semantic search`.
5. انتقل إلى [Notebook 07 — Evaluation and Error Analysis](07_evaluation_error_analysis.ipynb).
